# ML Pipeline Preparation
### 1. Import libraries and load data from database

In [1]:
import pickle
import os
import re
import sqlite3
import random
import nltk
import pandas as pd
import numpy as np
from nltk.tokenize import word_tokenize
from sqlalchemy import create_engine
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from nltk.stem.wordnet import WordNetLemmatizer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
random.seed(42)

/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


In [2]:
db_candidates = ['data/DisasterResponse.db', '../data/DisasterResponse.db']
db_path = next((p for p in db_candidates if os.path.exists(p)), db_candidates[0])
conn = sqlite3.connect(db_path)
df = pd.read_sql_query('SELECT * FROM ETL', conn)
conn.close()


In [3]:
df.head()

,id,message,original,genre,related,request,offer,aid_related,medical_help,medical_products,...,aid_centers,other_infrastructure,weather_related,floods,storm,fire,earthquake,cold,other_weather,direct_report
0,2,Weather update - a cold front from Cuba that c...,Un front froid se retrouve sur Cuba ce matin. ...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,Is the Hurricane over or is it not over,Cyclone nan fini osinon li pa fini,direct,1,0,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
2,8,Looking for someone but no name,"Patnm, di Maryani relem pou li banm nouvel li ...",direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9,UN reports Leogane 80-90 destroyed. Only Hospi...,UN reports Leogane 80-90 destroyed. Only Hospi...,direct,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,12,"says: west side of Haiti, rest of the country ...",facade ouest d Haiti et le reste du pays aujou...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
X = df['message']
Y = df.iloc[:,4:]
categories = Y.columns

In [5]:
Y.head()

,related,request,offer,aid_related,medical_help,medical_products,search_and_rescue,security,military,child_alone,...,aid_centers,other_infrastructure,weather_related,floods,storm,fire,earthquake,cold,other_weather,direct_report
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,1,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. Tokenization function

In [6]:
def tokenize(text):
    text = re.sub(r"[^a-zA-Z0-9]", " ", text.lower())
    try:
        words = word_tokenize(text)
    except LookupError:
        words = text.split()

    try:
        stop_words = set(stopwords.words("english"))
    except LookupError:
        stop_words = set()

    lemmatizer = WordNetLemmatizer()
    cleaned_words = []
    for word in words:
        if word in stop_words:
            continue
        try:
            clean_word = lemmatizer.lemmatize(word).strip()
        except LookupError:
            clean_word = word.strip()
        if clean_word:
            cleaned_words.append(clean_word)
    return cleaned_words


### 3. Build the ML pipeline

In [7]:
pipeline = Pipeline([
                    ('vect', CountVectorizer(tokenizer=tokenize)),
                    ('tfidf', TfidfTransformer()),
                    ('clf', MultiOutputClassifier(RandomForestClassifier()))
])

### 4. Train the pipeline

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('vect',
                 CountVectorizer(tokenizer=<function tokenize at 0x7fb860614e50>)),
                ('tfidf', TfidfTransformer()),
                ('clf',
                 MultiOutputClassifier(estimator=RandomForestClassifier()))])

### 5. Evaluate the model

In [9]:
y_pred = pipeline.predict(X_test)
target_names = list(y_test.columns[1:])
print(classification_report(y_test.iloc[:, 1:].values, np.array([x[1:] for x in y_pred]), target_names=target_names))

                        precision    recall  f1-score   support

               request       0.83      0.48      0.61       886
                 offer       0.00      0.00      0.00        26
           aid_related       0.78      0.68      0.72      2186
          medical_help       0.65      0.07      0.13       385
      medical_products       0.71      0.08      0.14       276
     search_and_rescue       0.67      0.07      0.12       150
              security       0.00      0.00      0.00        83
              military       0.71      0.03      0.05       179
           child_alone       0.00      0.00      0.00         0
                 water       0.85      0.40      0.54       323
                  food       0.85      0.59      0.70       564
               shelter       0.84      0.34      0.49       459
              clothing       0.75      0.12      0.20        77
                 money       1.00      0.04      0.08       120
        missing_people       0.00      

/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/tobibolu/op

### 6. Optimize with GridSearch

In [10]:
pipeline.get_params()

{'memory': None,
 'steps': [('vect',
   CountVectorizer(tokenizer=<function tokenize at 0x7fb860614e50>)),
  ('tfidf', TfidfTransformer()),
  ('clf', MultiOutputClassifier(estimator=RandomForestClassifier()))],
 'verbose': False,
 'vect': CountVectorizer(tokenizer=<function tokenize at 0x7fb860614e50>),
 'tfidf': TfidfTransformer(),
 'clf': MultiOutputClassifier(estimator=RandomForestClassifier()),
 'vect__analyzer': 'word',
 'vect__binary': False,
 'vect__decode_error': 'strict',
 'vect__dtype': numpy.int64,
 'vect__encoding': 'utf-8',
 'vect__input': 'content',
 'vect__lowercase': True,
 'vect__max_df': 1.0,
 'vect__max_features': None,
 'vect__min_df': 1,
 'vect__ngram_range': (1, 1),
 'vect__preprocessor': None,
 'vect__stop_words': None,
 'vect__strip_accents': None,
 'vect__token_pattern': '(?u)\\b\\w\\w+\\b',
 'vect__tokenizer': <function __main__.tokenize(text)>,
 'vect__vocabulary': None,
 'tfidf__norm': 'l2',
 'tfidf__smooth_idf': True,
 'tfidf__sublinear_tf': False,
 'tfidf_

In [11]:
parameters = {'clf__estimator__min_samples_leaf':[1, 5, 10],
              'clf__estimator__n_jobs': [1],
              'clf__estimator__n_estimators': [10, 20, 50]
              }
cv = RandomizedSearchCV(pipeline, param_distributions=parameters, n_iter=9)

In [12]:
cv.fit(X_train, y_train)

RandomizedSearchCV(estimator=Pipeline(steps=[('vect',
                                              CountVectorizer(tokenizer=<function tokenize at 0x7fb860614e50>)),
                                             ('tfidf', TfidfTransformer()),
                                             ('clf',
                                              MultiOutputClassifier(estimator=RandomForestClassifier()))]),
                   n_iter=9,
                   param_distributions={'clf__estimator__min_samples_leaf': [1,
                                                                             5,
                                                                             10],
                                        'clf__estimator__n_estimators': [10, 20,
                                                                         50],
                                        'clf__estimator__n_jobs': [1]})

In [13]:
y_pred = cv.predict(X_test)
print(classification_report(y_test.iloc[:,1:].values, np.array([x[1:] for x in y_pred]), target_names=list(y_test.columns[1:])))

                        precision    recall  f1-score   support

               request       0.81      0.47      0.59       886
                 offer       0.00      0.00      0.00        26
           aid_related       0.78      0.66      0.72      2186
          medical_help       0.59      0.09      0.16       385
      medical_products       0.72      0.12      0.21       276
     search_and_rescue       0.69      0.07      0.13       150
              security       0.00      0.00      0.00        83
              military       0.73      0.04      0.08       179
           child_alone       0.00      0.00      0.00         0
                 water       0.83      0.47      0.60       323
                  food       0.87      0.54      0.67       564
               shelter       0.81      0.37      0.51       459
              clothing       0.79      0.14      0.24        77
                 money       0.83      0.04      0.08       120
        missing_people       0.00      

/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/tobibolu/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/tobibolu/op

### 7. Evaluate the optimized model

### 8. Export the model as a pickle file

In [14]:
filename = 'classifier.sav'
pickle.dump(pipeline, open(filename, 'wb'))